In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.multitest import multipletests
# =========================================
# Univariate logistic regressions (ORs)
# Actigraphy sample, unbinned dataset
# Includes FDR correction (BH)
# =========================================

in_path = "../data/mcs_sh_sample_Feb_28_OR_preprocessed.csv"
out_all = "../results/or/univariate_odds_ratios_95CI_actigraphy_only_ALL.csv"
out_sig = "../results/or/univariate_odds_ratios_95CI_actigraphy_only_SIG.csv"

target_col = "suicide_17y"
do_scale = True  # OR per 1 SD for numeric predictors

df = pd.read_csv(in_path)
df.columns = df.columns.str.strip()

df = df.replace(["", " ", "NA", "N/A", "null", "nan"], np.nan)

# Extra safety: ensure actigraphy present
if "mean_acc_24h" in df.columns:
    df = df[df["mean_acc_24h"].notna()].copy()

# Target mapping
df[target_col] = df[target_col].astype(str).str.strip().str.lower().map({"no": 0, "yes": 1})
df = df.dropna(subset=[target_col]).copy()

y = df[target_col].astype(int).values.astype(np.float64)

# Features
X = df.drop(columns=[target_col]).copy()

# Enforce reference category ordering for dummy coding
if "child_alcohol" in X.columns:
    X["child_alcohol"] = pd.Categorical(X["child_alcohol"], ["none","some","many"])

if "child_cannabis" in X.columns:
    X["child_cannabis"] = pd.Categorical(X["child_cannabis"], ["Never","one to four","more than 5"])

if "mAlcohol_binary" in X.columns:
    X["mAlcohol_binary"] = pd.Categorical(X["mAlcohol_binary"], ["low risk","high risk"])

if "fAlcohol_binary" in X.columns:
    X["fAlcohol_binary"] = pd.Categorical(X["fAlcohol_binary"], ["low risk","high risk"])

if "mEdu" in X.columns:
    X["mEdu"] = pd.Categorical(X["mEdu"], ["lower edu","higher edu","overseas"])

# One hot encode remaining categoricals
cat_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
if len(cat_cols) > 0:
    X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

# Force numeric, handle inf, impute
X = X.apply(pd.to_numeric, errors="coerce")
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median(numeric_only=True))

# Drop constant columns
constant_cols = X.columns[X.nunique(dropna=False) <= 1]
if len(constant_cols) > 0:
    X = X.drop(columns=constant_cols)

# Decide which columns to scale: only genuinely continuous ones
# Heuristic: scale if more than 5 unique values
scale_cols = [c for c in X.columns if X[c].nunique() > 5]

results = []

for col in X.columns:
    x = X[[col]].copy()

    # Skip if constant
    if x[col].nunique(dropna=False) <= 1:
        continue

    # Optional scaling only for continuous like vars
    if do_scale and col in scale_cols:
        scaler = StandardScaler()
        x[col] = scaler.fit_transform(x[[col]])

    # HARD CAST to float numpy arrays for statsmodels
    exog = sm.add_constant(x, has_constant="add").to_numpy(dtype=np.float64)
    endog = y  # already float64

    try:
        model = sm.Logit(endog, exog).fit(disp=0, maxiter=200)

        beta = model.params[1]  # index 0 is const, index 1 is predictor
        se = model.bse[1]
        pval = model.pvalues[1]

        ci_low = beta - 1.96 * se
        ci_high = beta + 1.96 * se

        results.append({
            "Feature": col,
            "Beta": float(beta),
            "OR": float(np.exp(beta)),
            "CI_low": float(np.exp(ci_low)),
            "CI_high": float(np.exp(ci_high)),
            "p_value": float(pval),
            "N": int(len(endog)),
            "Error": ""
        })

    except Exception as e:
        results.append({
            "Feature": col,
            "Beta": np.nan,
            "OR": np.nan,
            "CI_low": np.nan,
            "CI_high": np.nan,
            "p_value": np.nan,
            "N": int(len(endog)),
            "Error": str(e)
        })

or_table = pd.DataFrame(results)

# FDR correction (BH), only on valid p values
valid_mask = or_table["p_value"].notna()
pvals = or_table.loc[valid_mask, "p_value"].values

if len(pvals) > 0:
    _, pvals_fdr, _, _ = multipletests(pvals, alpha=0.05, method="fdr_bh")
    or_table.loc[valid_mask, "p_value_fdr"] = pvals_fdr
    or_table.loc[valid_mask, "significant_fdr"] = pvals_fdr < 0.05
else:
    or_table["p_value_fdr"] = np.nan
    or_table["significant_fdr"] = False

or_table_sorted = or_table.sort_values("p_value", ascending=True, na_position="last")
or_table_sorted.to_csv(out_all, index=False)

sig = or_table_sorted[or_table_sorted["significant_fdr"] == True].copy()
sig.to_csv(out_sig, index=False)

print(or_table_sorted.head(30))
print(f"Saved full univariate OR table: {out_all}")
print(f"Saved FDR significant univariate OR table: {out_sig}")

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler

# =========================================
# Multivariate logistic regression (aORs)
# Actigraphy sample, unbinned dataset
# Predictors are prespecified
# =========================================

in_path = "../data/mcs_sh_sample_Feb_28_OR_preprocessed.csv"
out_path = "../results/or/multivariate_odds_ratios_95CI_actigraphy_only.csv"

target_col = "suicide_17y"

predictors = [
    "sex",
    "FHYPER", "FCONDUCT", "FPEER", "FPROSOC", "FEMOTION",
    "SelfEsteem", "Depression",
    "child_alcohol", "child_cannabis",
    "child_ADHD", "child_autism", "child_specneeds",
    "mvpa_acc_5sec", "l5_hour_start", "m5_hour_start", "mean_acc_24h"
]

# Optional: scale continuous predictors so ORs are per 1 SD.
# Dummies created from categoricals are NOT scaled.
do_scale_continuous = True

df = pd.read_csv(in_path)
df.columns = df.columns.str.strip()

# Normalize missing markers
df = df.replace(["", " ", "NA", "N/A", "null", "None", "nan"], np.nan)

# Safety check: ensure actigraphy present
if "mean_acc_24h" in df.columns:
    df = df[df["mean_acc_24h"].notna()].copy()

# Target mapping
df[target_col] = df[target_col].astype(str).str.strip().str.lower().map({"no": 0, "yes": 1})
df = df.dropna(subset=[target_col]).copy()
y = df[target_col].astype(int).to_numpy(dtype=np.float64)

# Keep only columns needed (fail loudly if something is missing)
missing = [c for c in predictors if c not in df.columns]
if len(missing) > 0:
    raise ValueError(f"These predictors are missing from the dataset: {missing}")

X = df[predictors].copy()

# Enforce reference category ordering for dummy coding
if "child_alcohol" in X.columns:
    X["child_alcohol"] = pd.Categorical(X["child_alcohol"], ["none","some","many"])

if "child_cannabis" in X.columns:
    X["child_cannabis"] = pd.Categorical(X["child_cannabis"], ["Never","one to four","more than 5"])

if "mAlcohol_binary" in X.columns:
    X["mAlcohol_binary"] = pd.Categorical(X["mAlcohol_binary"], ["low risk","high risk"])

if "fAlcohol_binary" in X.columns:
    X["fAlcohol_binary"] = pd.Categorical(X["fAlcohol_binary"], ["low risk","high risk"])

if "mEdu" in X.columns:
    X["mEdu"] = pd.Categorical(X["mEdu"], ["lower edu","higher edu","overseas"])

# One hot encode categorical predictors (sex, child_* variables might be categorical)
cat_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
if len(cat_cols) > 0:
    X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

# Force numeric, handle inf, impute any remaining NaNs
X = X.apply(pd.to_numeric, errors="coerce")
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median(numeric_only=True))

# Drop constant columns
constant_cols = X.columns[X.nunique(dropna=False) <= 1]
if len(constant_cols) > 0:
    X = X.drop(columns=constant_cols)

# Scale only continuous-like columns (not binary dummies)
if do_scale_continuous:
    continuous_cols = [c for c in X.columns if X[c].nunique() > 5]
    if len(continuous_cols) > 0:
        scaler = StandardScaler()
        X.loc[:, continuous_cols] = scaler.fit_transform(X[continuous_cols])

# Build design matrix and hard-cast to float64 arrays (prevents dtype object errors)
X_sm = sm.add_constant(X, has_constant="add").to_numpy(dtype=np.float64)

# Fit model
result = sm.Logit(y, X_sm).fit(disp=1, maxiter=200)

# Build OR table
params = result.params
bse = result.bse
pvals = result.pvalues

ci_low = params - 1.96 * bse
ci_high = params + 1.96 * bse

feature_names = ["const"] + list(X.columns)

or_table = pd.DataFrame({
    "Feature": feature_names,
    "Beta": params,
    "OR": np.exp(params),
    "CI_low": np.exp(ci_low),
    "CI_high": np.exp(ci_high),
    "p_value": pvals
})

or_table = or_table[or_table["Feature"] != "const"].sort_values("p_value").reset_index(drop=True)

print(or_table)
or_table.to_csv(out_path, index=False)
print(f"Saved multivariate OR table: {out_path}")
print(f"N used: {len(y)}, Events: {int(y.sum())}")

In [ ]:
# =========================================
# Model fit statistics
# =========================================

ll_model = result.llf      # log-likelihood of fitted model
ll_null = result.llnull    # log-likelihood of null model
n = int(result.nobs)

# McFadden pseudo R²
r2_mcfadden = 1 - (ll_model / ll_null)

# Cox & Snell pseudo R²
r2_cox_snell = 1 - np.exp((2 / n) * (ll_null - ll_model))

# Nagelkerke pseudo R²
r2_nagelkerke = r2_cox_snell / (1 - np.exp((2 / n) * ll_null))

# Likelihood ratio test
lr_stat = result.llr
lr_pvalue = result.llr_pvalue

# AIC and BIC
aic = result.aic
bic = result.bic

print("\nModel fit statistics:")
print(f"Nagelkerke R²: {r2_nagelkerke:.3f}")
print(f"Cox & Snell R²: {r2_cox_snell:.3f}")
print(f"McFadden R²: {r2_mcfadden:.3f}")
print(f"Likelihood ratio χ²: {lr_stat:.2f}, p = {lr_pvalue:.3g}")
print(f"AIC: {aic:.2f}")
print(f"BIC: {bic:.2f}")